In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
file_path = '/content/drive/MyDrive/dengue_thesis/Temporal_extract_V1_3.csv'
# file_path = 'Temporal_extract_V1_3.csv'

In [4]:
chunks = pd.read_csv(
    file_path,
    chunksize=200000,
    usecols=['adm_0_name', 'T_res', 'Year',
             'dengue_total', 'calendar_start_date', 'S_res']
)

target_chunks = []
for chunk in chunks:
    filtered = chunk[
        (chunk['T_res'] == 'Week') &
        (chunk['adm_0_name'].isin(['SRI LANKA'])) &
        (chunk['S_res'] == 'Admin0')
    ]
    if len(filtered) > 0:
        target_chunks.append(filtered)

df = pd.concat(target_chunks).sort_values('calendar_start_date')
print('Done! Rows found:', len(df))
print(df['adm_0_name'].value_counts())

Done! Rows found: 641
adm_0_name
SRI LANKA    641
Name: count, dtype: int64


In [5]:
slk = df[
    (df['adm_0_name'] == 'SRI LANKA') &
    (df['Year'] == 2017)
].sort_values('calendar_start_date').reset_index(drop=True)

print('Sri Lanka 2017 weeks:', len(slk))
print(slk[['calendar_start_date', 'dengue_total']])

Sri Lanka 2017 weeks: 51
   calendar_start_date  dengue_total
0           2017-01-08          2426
1           2017-01-15          2841
2           2017-01-22          2558
3           2017-01-29          2358
4           2017-02-05          2063
5           2017-02-12          2278
6           2017-02-19          1801
7           2017-02-26          2403
8           2017-03-05          2684
9           2017-03-12          2557
10          2017-03-19          2931
11          2017-03-26          2892
12          2017-04-02          2946
13          2017-04-09          1973
14          2017-04-16          4043
15          2017-04-23          3558
16          2017-04-30          3867
17          2017-05-07          3225
18          2017-05-14          4681
19          2017-05-21          4233
20          2017-05-28          4221
21          2017-06-04          4432
22          2017-06-11          4688
23          2017-06-18          6331
24          2017-06-25          5665
25          2

In [6]:
total_affected = slk['dengue_total'].sum()
print('Total dengue cases in 2017:', total_affected)

Total dengue cases in 2017: 176272


In [7]:
N_slk = slk['dengue_total'].sum()/0.1

new_cases = slk['dengue_total'].values.astype(float)
T = len(new_cases)

time      = np.arange(T, dtype=float)
I_raw     = new_cases.copy()
R_raw     = np.concatenate([[0], np.cumsum(new_cases[:-1])])
S_raw     = N_slk - I_raw - R_raw

slk_sir = pd.DataFrame({
    'time':        time,
    'Susceptible': S_raw / N_slk,
    'Infected':    I_raw / N_slk,
    'Recovered':   R_raw / N_slk
})

print(f"Total Populations: {N_slk}")
print(slk_sir)


Total Populations: 1762720.0
    time  Susceptible  Infected  Recovered
0    0.0     0.998624  0.001376   0.000000
1    1.0     0.997012  0.001612   0.001376
2    2.0     0.995561  0.001451   0.002988
3    3.0     0.994223  0.001338   0.004439
4    4.0     0.993053  0.001170   0.005777
5    5.0     0.991760  0.001292   0.006947
6    6.0     0.990739  0.001022   0.008240
7    7.0     0.989376  0.001363   0.009261
8    8.0     0.987853  0.001523   0.010624
9    9.0     0.986402  0.001451   0.012147
10  10.0     0.984739  0.001663   0.013598
11  11.0     0.983099  0.001641   0.015261
12  12.0     0.981428  0.001671   0.016901
13  13.0     0.980308  0.001119   0.018572
14  14.0     0.978015  0.002294   0.019692
15  15.0     0.975996  0.002018   0.021985
16  16.0     0.973802  0.002194   0.024004
17  17.0     0.971973  0.001830   0.026198
18  18.0     0.969317  0.002656   0.028027
19  19.0     0.966916  0.002401   0.030683
20  20.0     0.964521  0.002395   0.033084
21  21.0     0.962007  0.

In [8]:
slk_sir.to_csv('/content/drive/MyDrive/dengue_thesis/dengue_srilanka_2017.csv', index=False)
print('Drive-এ saved!')

Drive-এ saved!
